# ERA5 and ERA5-Land Grid Alignment Verification
This notebook checks whether the ERA5 and ERA5-Land exports use matching spatial grid-cell grometires and timestamps. 

## What this norebook verifies
1. Both files contain the required columns.
2. Both files contain the same set of grid_id labels.
3. The same grid_id may refer to more than one geometry, so grid_id is not treated as the final spatial key.
4. Polygon geometries are standardised and converted into deterministic geometry_id hashes.
5. Both datasets contain the same geometry set.
6. Both datasets contain the same geometry-datetime combinations. 

## Scope
The default file names below refer to *January 2018* exports. Update the paths to reuse this notebook with other months or years.

## 1. Imports and file configuration
Keep both Excel files in the same folder as this notebook, or replace tha paths below with their full locations. 

In [1]:
from pathlib import Path
import hashlib
import json

import pandas as pd

# Update these paths when checking different exports.
ERA5_FILE = Path("FireFusion_ERA5_Victoria_12Hourly_5kmGrid_Jan2018.xlsx")
ERA5_LAND_FILE = Path("2018_Jan_ERA5Land_Victoria_12Hourly_5kmGrid.xlsx")

REQUIRED_COLUMNS = {"grid_id", ".geo", "datetime"}

## 2. Load the datasets

The cell checks that both files exist before reading them. This gives a clear error message if a file name or location is incorrect.

In [2]:
def load_excel_dataset(path: Path, dataset_name: str) -> pd.DataFrame:
    """Load an Excel dataset and raise a clear error if the file is missing."""
    if not path.exists():
        raise FileNotFoundError(
            f"{dataset_name} file was not found: {path.resolve()}\n"
            "Move the file beside this notebook or update the file path."
        )

    dataframe = pd.read_excel(path)
    print(f"{dataset_name}: {len(dataframe):,} rows loaded")
    return dataframe


era5 = load_excel_dataset(ERA5_FILE, "ERA5")
era5_land = load_excel_dataset(ERA5_LAND_FILE, "ERA5-Land")

ERA5: 883,996 rows loaded
ERA5-Land: 883,996 rows loaded


## 3. Validate the required columns

The comparison requires:

- `grid_id`: the original Earth Engine feature identifier
- `.geo`: the exported polygon geometry
- `datetime`: the 12-hour observation time


In [3]:
def validate_columns(
    dataframe: pd.DataFrame,
    dataset_name: str,
    required_columns: set[str],
) -> None:
    """Confirm that all required columns are available."""
    missing = required_columns - set(dataframe.columns)

    if missing:
        raise ValueError(
            f"{dataset_name} is missing required columns: {sorted(missing)}"
        )

    print(f"{dataset_name}: all required columns are available")


validate_columns(era5, "ERA5", REQUIRED_COLUMNS)
validate_columns(era5_land, "ERA5-Land", REQUIRED_COLUMNS)

print("\nERA5 columns:")
print(era5.columns.tolist())

print("\nERA5-Land columns:")
print(era5_land.columns.tolist())

ERA5: all required columns are available
ERA5-Land: all required columns are available

ERA5 columns:
['system:index', 'count', 'datetime', 'dewpoint_temperature_2m_c', 'grid_id', 'interval_end', 'interval_start', 'label', 'surface_pressure', 'temperature_2m_c', 'timestamp', 'total_precipitation', 'u_component_of_wind_10m', 'v_component_of_wind_10m', 'wind_speed_10m', '.geo']

ERA5-Land columns:
['system:index', 'count', 'datetime', 'grid_id', 'interval_end', 'interval_start', 'label', 'skin_temperature_c', 'soil_temperature_level_1_c', 'surface_solar_radiation_downwards', 'surface_thermal_radiation_downwards', 'temperature_2m_c', 'timestamp', 'u_component_of_wind_10m', 'v_component_of_wind_10m', '.geo']


## 4. Compare the original `grid_id` labels

This confirms whether both exports contain the same set of `grid_id` values.

**Important:** Matching IDs alone do not prove that each ID identifies one unique spatial cell. The later geometry checks provide the stronger verification.

In [4]:
era5_grid_ids = set(era5["grid_id"].dropna().astype(str))
era5_land_grid_ids = set(era5_land["grid_id"].dropna().astype(str))

grid_ids_match = era5_grid_ids == era5_land_grid_ids

print(f"ERA5 unique grid IDs: {len(era5_grid_ids):,}")
print(f"ERA5-Land unique grid IDs: {len(era5_land_grid_ids):,}")
print(f"Do the grid ID sets match? {grid_ids_match}")
print(f"Only in ERA5: {len(era5_grid_ids - era5_land_grid_ids):,}")
print(f"Only in ERA5-Land: {len(era5_land_grid_ids - era5_grid_ids):,}")

ERA5 unique grid IDs: 216
ERA5-Land unique grid IDs: 216
Do the grid ID sets match? True
Only in ERA5: 0
Only in ERA5-Land: 0


## 5. Check whether each `grid_id` maps to one geometry

This diagnostic shows whether the original `grid_id` can be used safely as a unique spatial identifier.

If many IDs have multiple geometries, do not merge using `grid_id` alone.

In [5]:
era5_geometry_counts = era5.groupby("grid_id")[".geo"].nunique(dropna=True)
era5_land_geometry_counts = (
    era5_land.groupby("grid_id")[".geo"].nunique(dropna=True)
)

era5_multi_geometry_ids = era5_geometry_counts[era5_geometry_counts > 1]
era5_land_multi_geometry_ids = (
    era5_land_geometry_counts[era5_land_geometry_counts > 1]
)

print(
    "ERA5 grid IDs linked to multiple geometries:",
    len(era5_multi_geometry_ids),
)
print(
    "ERA5-Land grid IDs linked to multiple geometries:",
    len(era5_land_multi_geometry_ids),
)

# Display only a small sample to keep the notebook readable.
print("\nERA5 sample:")
print(era5_multi_geometry_ids.head(10))

print("\nERA5-Land sample:")
print(era5_land_multi_geometry_ids.head(10))

ERA5 grid IDs linked to multiple geometries: 214
ERA5-Land grid IDs linked to multiple geometries: 214

ERA5 sample:
grid_id
2221     2
2222     5
2223    14
2224    13
2225    14
2226    14
2227    14
2228    14
2229    15
2230    15
Name: .geo, dtype: int64

ERA5-Land sample:
grid_id
2221     2
2222     5
2223    14
2224    13
2225    14
2226    14
2227    14
2228    14
2229    15
2230    15
Name: .geo, dtype: int64


## 6. Create temporary geometry hashes for comparison

The exported `.geo` polygons are converted into deterministic hashes to make the geometry comparison easier.

These hashes are created only for this verification and are not part of the original datasets.

In [6]:
def canonicalise_geometry(value: object) -> str | None:
    """Return a standard JSON representation of an exported geometry."""
    if pd.isna(value):
        return None

    try:
        geometry = json.loads(str(value))
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid geometry JSON: {value}") from exc

    return json.dumps(
        geometry,
        sort_keys=True,
        separators=(",", ":"),
    )


def create_geometry_id(value: object) -> str | None:
    """Create a deterministic SHA-256 identifier from geometry JSON."""
    canonical_geometry = canonicalise_geometry(value)

    if canonical_geometry is None:
        return None

    return hashlib.sha256(
        canonical_geometry.encode("utf-8")
    ).hexdigest()


era5 = era5.copy()
era5_land = era5_land.copy()

era5["geometry_id"] = era5[".geo"].apply(create_geometry_id)
era5_land["geometry_id"] = era5_land[".geo"].apply(create_geometry_id)

print("Geometry IDs created successfully.")

Geometry IDs created successfully.


## 7. Compare the unique geometry sets

This is the main spatial-alignment check. It verifies that every polygon geometry in one export also exists in the other.

In [7]:
era5_geometries = set(era5["geometry_id"].dropna())
era5_land_geometries = set(era5_land["geometry_id"].dropna())

geometry_sets_match = era5_geometries == era5_land_geometries
only_in_era5 = era5_geometries - era5_land_geometries
only_in_era5_land = era5_land_geometries - era5_geometries

print(f"ERA5 unique geometries: {len(era5_geometries):,}")
print(f"ERA5-Land unique geometries: {len(era5_land_geometries):,}")
print(f"Do both datasets contain the same geometries? {geometry_sets_match}")
print(f"Only in ERA5: {len(only_in_era5):,}")
print(f"Only in ERA5-Land: {len(only_in_era5_land):,}")

ERA5 unique geometries: 14,258
ERA5-Land unique geometries: 14,258
Do both datasets contain the same geometries? True
Only in ERA5: 0
Only in ERA5-Land: 0


## 8. Compare geometry–datetime combinations

Matching geometry sets show that the same spatial cells exist in both files.

This additional check confirms that the same cells also appear at the same 12-hour timestamps.

In [8]:
def create_spatiotemporal_keys(dataframe: pd.DataFrame) -> set[tuple[str, str]]:
    """Return unique (datetime, geometry_id) combinations."""
    valid_rows = dataframe[["datetime", "geometry_id"]].dropna()

    return set(
        zip(
            valid_rows["datetime"].astype(str),
            valid_rows["geometry_id"].astype(str),
        )
    )


era5_time_grid = create_spatiotemporal_keys(era5)
era5_land_time_grid = create_spatiotemporal_keys(era5_land)

spatiotemporal_keys_match = era5_time_grid == era5_land_time_grid

print(f"ERA5 geometry–datetime combinations: {len(era5_time_grid):,}")
print(
    "ERA5-Land geometry–datetime combinations:",
    f"{len(era5_land_time_grid):,}",
)
print(
    "Do geometry and datetime combinations match?",
    spatiotemporal_keys_match,
)
print(
    "Only in ERA5:",
    len(era5_time_grid - era5_land_time_grid),
)
print(
    "Only in ERA5-Land:",
    len(era5_land_time_grid - era5_time_grid),
)

ERA5 geometry–datetime combinations: 883,996
ERA5-Land geometry–datetime combinations: 883,996
Do geometry and datetime combinations match? True
Only in ERA5: 0
Only in ERA5-Land: 0


## 9. Final verification summary

The final status uses the geometry and geometry–datetime checks as the main evidence.

The original `grid_id` comparison is reported separately because it is useful for diagnostics but is not a sufficient spatial verification by itself.

In [9]:
checks = {
    "Original grid ID sets match": grid_ids_match,
    "Geometry sets match": geometry_sets_match,
    "Geometry–datetime combinations match": spatiotemporal_keys_match,
    "No geometry exists only in ERA5": len(only_in_era5) == 0,
    "No geometry exists only in ERA5-Land": len(only_in_era5_land) == 0,
}

summary = pd.DataFrame(
    {
        "Check": checks.keys(),
        "Passed": checks.values(),
    }
)

display(summary)

spatial_alignment_passed = all(
    [
        geometry_sets_match,
        spatiotemporal_keys_match,
        len(only_in_era5) == 0,
        len(only_in_era5_land) == 0,
    ]
)

if spatial_alignment_passed:
    print(
        "\nPASS: The ERA5 and ERA5-Land exports contain matching "
        "grid-cell geometries and matching geometry–datetime combinations."
    )
else:
    print(
        "\nFAIL: Differences were found between the ERA5 and ERA5-Land "
        "spatial or temporal coverage."
    )

,Check,Passed
0,Original grid ID sets match,True
1,Geometry sets match,True
2,Geometry–datetime combinations match,True
3,No geometry exists only in ERA5,True
4,No geometry exists only in ERA5-Land,True



PASS: The ERA5 and ERA5-Land exports contain matching grid-cell geometries and matching geometry–datetime combinations.


## Key Findings and Recommendations

- The January 2018 ERA5 and ERA5-Land exports contain the same polygon geometries.
- The geometry and datetime combinations also match across both datasets.
- The existing `grid_id` values are not unique spatial identifiers because one `grid_id` can be associated with multiple geometries.
- Therefore, `grid_id` should not be used by itself when merging the datasets.
- The `.geo` column provides the actual spatial geometry and should be used to validate alignment.
- For repeatable merging, the team should create a stable spatial identifier from the geometry, such as a canonical geometry hash or consistent grid-centroid coordinates.

## Conclusion

For the files tested, the verification compares the complete polygon geometry sets and their associated timestamps.

A successful result supports the following statement:

> The January 2018 ERA5 and ERA5-Land exports contain matching grid-cell geometries and matching 12-hour geometry–datetime combinations.

### Interpretation of `grid_id`

The original Earth Engine `grid_id` values should not be treated as the only spatial key when one ID is linked to multiple geometries. For reproducible comparisons or merges, it is better to use a stable spatial identifier derived from the geometry together with `datetime`, rather than relying on the existing `grid_id` alone.

### Reuse

To check another period:

1. Update `ERA5_FILE` and `ERA5_LAND_FILE`.
2. Restart the kernel.
3. Run all cells from top to bottom.
4. Review the final PASS/FAIL summary.